In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [102]:
x_train = np.array([[1, 3], [2, 4], [6, 8], [5, 7], [10, 12]])
y_train = np.array([0, 1, 1, 0, 1])
# y_train = np.array([[1, 0], [0, 1], [0, 1], [1, 0]])

# [odd, even]

# x_train = np.array([[1, 2, 3, 4, 5, 6]])
# y_train = np.array([1, 0, 1, 0, 1, 0])

n0 = x_train.shape[1]
print(n0)
n1 = 5
n2 = 5
n3 = 2

2


In [103]:
np.random.seed(69)
X_values = np.array([1, 2, 3, 4])

weights = np.random.randn(5,4)
biases = np.random.randn(1,n1)

Transposing weights is necessary to multiply rows by rows because the dot product multiplies rows by columns. 

Here, the output 3x3 are the output of the weight + bias of the each node (n1 = 3). So the first row of the array are the value 3 neurons from the 2 input 1, 2

In [104]:
outputs = np.dot(X_values, weights.T) + biases
print(outputs)

[[  1.72427991   5.87079497   0.20799227  -4.07516954 -10.86097873]]


In [105]:
class Layer:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01*np.random.randn(n_inputs, n_neurons) # 0.1 is a scaling factor for weights to be small. 
        # self.weights = self.weights.T                           #or we can just do (n_inputs, n_neurons) instead of (n_neurons, n_inputs) to eliminate this
        self.biases = np.zeros((1, n_neurons))
        
    def ReLU(self, inputs):
        self.reLUinputs = inputs
        self.output = np.maximum(0, inputs)
    
    def softmax(self, inputs):
        self.exponential_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = self.exponential_values / np.sum(self.exponential_values, axis=1, keepdims=True) #axis=1 is for row-wise sum and keepdims=True is to keep the dimensions of the output same as input
    
    def forward(self, inputs):
        self.inputs = inputs
        self.forward_output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)
    
    def ReLUDerivative(self, dvalues):
        self.dreLU = dvalues.copy()
        self.dreLU[self.reLUinputs <= 0] = 0
    
    def softmaxDerivative(self, dvalues, y_train):
        samples = len(dvalues)

        #removing one hot encoding
        if len(y_train.shape) == 2:
            y_train = np.argmax(y_train, axis=1)
        
        self.dsoftmax = dvalues.copy()
        self.dsoftmax[range(samples), y_train] -= 1

        #normalizing the gradient
        self.dsoftmax /= samples

In [106]:
layer1 = Layer(n0, n1)
layer2 = Layer(n1, n2)
layer3 = Layer(n2, n3)

layer1.forward(x_train)
layer1.ReLU(layer1.forward_output)

print(layer1.forward_output)

layer2.forward(layer1.output)
layer2.ReLU(layer2.forward_output)

# print(layer2.output)


[[-0.04075019  0.04943916 -0.01041613 -0.01936682 -0.03183819]
 [-0.06229273  0.06704305 -0.02787893 -0.02569473 -0.04043635]
 [-0.1484629   0.13745863 -0.09773012 -0.05100635 -0.07482899]
 [-0.12692036  0.11985474 -0.08026733 -0.04467845 -0.06623083]
 [-0.23463306  0.20787422 -0.16758132 -0.07631798 -0.10922163]]


Why softmax activation function for the last neuron?
    - Because while calculating the probability of a classification, we need to use a function that can generate the classification. We cannot directly use sum of the previous output as the negative values will cancel out. We also cannot use ReLU (or absolute value) because we need the negative numbers to be able to back propagate and optimise our model. So we use Softmax Activation (just applying exponential) since exponential is always positive but does not lose the meaning of the negative number

In [107]:
layer3.forward(layer2.output)
print("Layer 3 output:")
print(layer3.forward_output)

layer3.softmax(layer3.forward_output)
print("Layer 3 exponential values:")
print(layer3.exponential_values)

print("Layer 3 softmax output:")
print(layer3.output)


Layer 3 output:
[[-2.79992662e-07 -6.30323493e-08]
 [-3.79690186e-07 -8.54763986e-08]
 [-7.78480279e-07 -1.75252596e-07]
 [-6.78782755e-07 -1.52808546e-07]
 [-1.17727037e-06 -2.65028793e-07]]
Layer 3 exponential values:
[[0.99999978 1.        ]
 [0.99999971 1.        ]
 [0.9999994  1.        ]
 [0.99999947 1.        ]
 [0.99999909 1.        ]]
Layer 3 softmax output:
[[0.49999995 0.50000005]
 [0.49999993 0.50000007]
 [0.49999985 0.50000015]
 [0.49999987 0.50000013]
 [0.49999977 0.50000023]]


In [108]:
print(layer3.output[range(len(layer3.output)), y_train])
log_values = -np.log(layer3.output[range(len(layer3.output)), y_train])
print(np.mean(log_values))

[0.49999995 0.50000007 0.50000015 0.49999987 0.50000023]
0.6931470738851329


In [109]:
class Loss:
    def calculateLoss(self, output, y):
        sample_losses = self.categoricalClassEntropy(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss

    def categoricalClassEntropy(self, y_pred, y_train):
        samples = len(y_pred)
        print(samples)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)
        
        print(f'y_pred_clipped: {y_pred_clipped}')
        print(f'y_train: {y_train}')

        #len of shape gives the number of dimensions of the array
        if len(y_train.shape) == 1: # scalar values [1, 0, 1] 
            print("Using scalar values")
            correct_confidences = y_pred_clipped[range(samples), y_train]

        elif len(y_train.shape) == 2: # one-hot encoded [ [0, 1, 0], [1, 0, 0], [0, 1, 0] ]
            print("Using one-hot encoded values")
            correct_confidences = np.sum(y_pred_clipped*y_train, axis=1)

        negative_log_likelihoods = -np.log(correct_confidences)
        print(f'correct_confidences: {correct_confidences}')
        print(f'negative_log_likelihoods: {negative_log_likelihoods}')
        return negative_log_likelihoods

    def CrossEntropylossDerivative(self, dvalues, y_train):
        samples = dvalues.shape[0]
        labels = dvalues.shape[1]

        if len(y_train.shape) == 1: # scalar values [1, 0, 1]
            y_train = np.eye(labels)[y_train]
            
        self.dloss = -y_train / dvalues
        self.dloss = self.dinputs / samples #normalizing


In [110]:
loss_function = Loss()
loss = loss_function.calculateLoss(layer3.output, y_train)
print(f'loss: {loss}')

predictions = np.argmax(layer3.output, axis=1)
print(predictions)

accuracy = np.mean(predictions == y_train)
print(f'accuracy: {accuracy}')

loss: 0.6931470738851329
[1 1 1 1 1]
accuracy: 0.6


In [111]:
print(layer3.output)
print(y_train)
layer3.softmaxDerivative(layer3.output, y_train)
print(layer3.dsoftmax)

[[0.49999995 0.50000005]
 [0.49999993 0.50000007]
 [0.49999985 0.50000015]
 [0.49999987 0.50000013]
 [0.49999977 0.50000023]]
[0 1 1 0 1]
[[-0.10000001  0.10000001]
 [ 0.09999999 -0.09999999]
 [ 0.09999997 -0.09999997]
 [-0.10000003  0.10000003]
 [ 0.09999995 -0.09999995]]


In [112]:
layer3.backward(layer3.dsoftmax)

layer2.ReLUDerivative(layer3.dinputs)
layer2.backward(layer2.dreLU)

layer1.ReLUDerivative(layer2.dinputs)
layer1.backward(layer1.dreLU)

print("Layer1 weights: ")
print(layer1.weights.shape)

print("Layer1 biases: ")
print(layer1.biases.shape)

print("Layer1 dweights: ")
print(layer1.dweights.shape)

print("Layer1 dbiases: ")
print(layer1.dbiases.shape)

Layer1 weights: 
(2, 5)
Layer1 biases: 
(1, 5)
Layer1 dweights: 
(2, 5)
Layer1 dbiases: 
(1, 5)
